# Notebook 00 — Setup & Smoke Test

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This notebook does two things:

1. Installs the five free, open-source packages the whole course uses (one cell).
2. Runs three tiny "smoke tests" so you can SEE that each piece works.

If every cell below runs without a red error, you're ready for Lesson 2. Nothing here
costs money — there are no accounts, API keys, or credit cards anywhere in this course.

## Step 1 — Install the packages

| Package | What it gives us |
|---|---|
| `numpy` | fast lists of numbers (our vectors live here) |
| `matplotlib` | drawing plots |
| `scikit-learn` | the PCA tool for seeing high-dimensional data in 2-D |
| `sentence-transformers` | the free HuggingFace model that turns text into vectors |
| `chromadb` | a local "vector store" that finds nearest vectors for us |

Run it once. In Colab you re-run it each new session — that's normal and takes ~a minute.

In [ ]:
# Run this first (Shift+Enter). Installs the five packages this course uses.
# -q means "quiet" so you don't see hundreds of lines of output.
%pip install -q numpy matplotlib scikit-learn sentence-transformers chromadb

print("Packages installed. On to the smoke tests.")

## Step 2.1 — NumPy + Matplotlib

The next cell makes a tiny set of numbers and draws four dots. If a little scatter plot
appears, your number-and-plotting tools work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Four dots, each with an (x, y) position.
points = np.array([[0.1, 0.2],
                   [0.4, 0.5],
                   [0.6, 0.1],
                   [0.9, 0.8]])
print("numpy version:", np.__version__)
print("shape of our data:", points.shape, "(4 dots, 2 numbers each)")

plt.figure(figsize=(4, 3))
plt.scatter(points[:, 0], points[:, 1])
plt.title("four dots")
plt.tight_layout()
plt.show()

print("NumPy + Matplotlib OK")

## Step 2.2 — The embedding model (sentence-transformers)

The next cell downloads a small, famous, free model called **all-MiniLM-L6-v2** the first
time you run it (~80 MB). The download happens **once**; after that it's instant.

> **Analogy:** think of this as installing a tiny dictionary of *meaning*. You hand it a
> sentence; it hands back a list of 384 numbers that stand for what the sentence means.

A one-time wait of a minute or two on the first run is completely normal.

In [ ]:
from sentence_transformers import SentenceTransformer

# Downloads the model the first time, then loads from cache after that.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Turn one word into a vector (a list of numbers).
vector = model.encode("hello")

print("how many numbers represent 'hello'? ->", len(vector))
print("the first five of them:", vector[:5])
print("\nsentence-transformers OK (expect 384 numbers)")

## Step 2.3 — The vector store (ChromaDB)

The last test stores three short sentences, then asks ChromaDB to find the one closest in
meaning to a question — using the same MiniLM model above. If it returns the cat sentence
for a cat question, everything is wired up correctly.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Use the SAME MiniLM model as the store's embedding function.
minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# An in-memory store (nothing saved to disk for this quick test).
client = chromadb.Client()
try:
    client.delete_collection("smoke_test")   # start clean if you re-run this cell
except Exception:
    pass
collection = client.create_collection(
    name="smoke_test",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},        # measure closeness by angle (cosine)
)

collection.add(
    ids=["s1", "s2", "s3"],
    documents=[
        "The cat slept on the warm windowsill.",
        "I drove my car to work this morning.",
        "She baked fresh bread for dinner.",
    ],
)

# Ask for the single closest sentence to a cat-flavoured question.
result = collection.query(query_texts=["a sleepy kitten"], n_results=1)

print("closest stored sentence:", result["documents"][0][0])
print("\nChromaDB OK")

## You're ready
If all three smoke tests ran without a red error box, your setup works and you can start
Lesson 2.

**If something went wrong:**
- *In Colab, a later cell says a name is "not defined":* the session reset. Re-run the
  install cell at the top, then run the cells in order from the start.
- *The model download is slow or stalls:* it's ~80 MB on first run; give it a minute, then
  re-run the cell.
- *`ModuleNotFoundError`:* the install cell didn't finish. Re-run Step 1, wait for it to finish,
  then continue.

Lesson 1 in the course walks through Colab, VS Code, and local Jupyter step by step.